# Airline Passenger Satisfaction

## 資料讀入
資料位置: /Users/apple/Desktop/Airline Passenger Satisfaction/train.csv

In [ ]:
import pandas as pd

In [53]:
df = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/train.csv")

## 確認資料欄位＆檢查

In [54]:
import pandas as pd
import numpy as np

## 載入套件

In [55]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression

In [56]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 25 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         103904 non-null  int64  
 1   id                                 103904 non-null  int64  
 2   Gender                             103904 non-null  object 
 3   Customer Type                      103904 non-null  object 
 4   Age                                103904 non-null  int64  
 5   Type of Travel                     103904 non-null  object 
 6   Class                              103904 non-null  object 
 7   Flight Distance                    103904 non-null  int64  
 8   Inflight wifi service              103904 non-null  int64  
 9   Departure/Arrival time convenient  103904 non-null  int64  
 10  Ease of Online booking             103904 non-null  int64  
 11  Gate location                      1039

In [57]:
df.describe()

,Unnamed: 0,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103904.000000,103594.000000
mean,51951.500000,64924.210502,39.379706,1189.448375,2.729683,3.060296,2.756901,2.976883,3.202129,3.250375,3.439396,3.358158,3.382363,3.351055,3.631833,3.304290,3.640428,3.286351,14.815618,15.178678
std,29994.645522,37463.812252,15.114964,997.147281,1.327829,1.525075,1.398929,1.277621,1.329533,1.349509,1.319088,1.332991,1.288354,1.315605,1.180903,1.265396,1.175663,1.312273,38.230901,38.698682
min,0.000000,1.000000,7.000000,31.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,25975.750000,32533.750000,27.000000,414.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,3.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,51951.500000,64856.500000,40.000000,843.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,3.000000,4.000000,3.000000,0.000000,0.000000
75%,77927.250000,97368.250000,51.000000,1743.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,5.000000,4.000000,5.000000,4.000000,12.000000,13.000000
max,103903.000000,129880.000000,85.000000,4983.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1592.000000,1584.000000


## 資料清洗

### 刪掉無用欄位

In [58]:
drop_cols = ["Unnamed: 0", "id"]
df = df.drop(columns=drop_cols, errors="ignore")

### 預覽資料

In [59]:
print("資料形狀:", df.shape)
print(df.head())
print(df.info())

資料形狀: (103904, 23)
   Gender      Customer Type  Age   Type of Travel     Class  Flight Distance  \
0    Male     Loyal Customer   13  Personal Travel  Eco Plus              460   
1    Male  disloyal Customer   25  Business travel  Business              235   
2  Female     Loyal Customer   26  Business travel  Business             1142   
3  Female     Loyal Customer   25  Business travel  Business              562   
4    Male     Loyal Customer   61  Business travel  Business              214   

   Inflight wifi service  Departure/Arrival time convenient  \
0                      3                                  4   
1                      3                                  2   
2                      2                                  2   
3                      2                                  5   
4                      3                                  3   

   Ease of Online booking  Gate location  ...  Inflight entertainment  \
0                       3              1  

## 處理目標變數 y

In [60]:
# satisfaction: "satisfied" / "neutral or dissatisfied"
df["satisfaction"] = df["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

In [61]:

print("\nsatisfaction 分布：")
print(df["satisfaction"].value_counts())



satisfaction 分布：
satisfaction
0    58879
1    45025
Name: count, dtype: int64


## 處理缺值

In [62]:
# Arrival Delay in Minutes 有缺值，用中位數填補
if "Arrival Delay in Minutes" in df.columns:
    df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(
        df["Arrival Delay in Minutes"].median()
    )

## 分出 X / y

In [63]:
X = df.drop("satisfaction", axis=1)
y = df["satisfaction"]

### 分辨欄位變數//類別變數 or 數值變數

In [64]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("\n類別欄位:", categorical_features)
print("數值欄位:", numeric_features)


類別欄位: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
數值欄位: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


## 資料預處理

In [65]:
# 類別欄位：補最常見值 + OneHot

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [66]:
# 數值欄位： Logistic Regression 會需要 scale

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## ColumnTransformer

In [67]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 切分訓練集 / 測試集

In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [69]:

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (83123, 22)
X_test shape: (20781, 22)
y_train shape: (83123,)
y_test shape: (20781,)


## Logistic Regression pipeline

In [70]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

## Logistic Regression Model Traing

In [71]:
logistic_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Flight Distance',
                                                   'Inflight wifi service',
                                                   'Departure/Arrival time '
                                                   'convenient',
                                                   'Ease of Online booking',
                                                   'Gate location',
                                                   'Food and drink',
                                                   'Online boarding',
                                                   'Seat comfort',
                                                   'Inflight entertainment'...
                                                   'Leg room service',
                                                   'Baggage handling',
                                                   'Checkin service',
                                                   'Inflight service',
                                                   'Cleanliness',
                                                   'Departure Delay in Minutes',
                                                   'Arrival Delay in Minutes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Gender', 'Customer Type',
                                                   'Type of Travel',
                                                   'Class'])])),
                ('model', LogisticRegression(max_iter=2000))])

## 預測與評估

In [75]:
y_pred = logistic_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8764255810596218

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89     11776
           1       0.87      0.84      0.86      9005

    accuracy                           0.88     20781
   macro avg       0.88      0.87      0.87     20781
weighted avg       0.88      0.88      0.88     20781


Confusion Matrix:
[[10633  1143]
 [ 1425  7580]]


## 取出欄位名稱 coefficient

In [76]:
feature_names_num = numeric_features

feature_names_cat = logistic_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

feature_names = list(feature_names_num) + list(feature_names_cat)

coefficients = logistic_model.named_steps["model"].coef_[0]


In [77]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values(by="coefficient", ascending=False)

In [78]:
print("正向影響最大的前 10 個變數：")
print(coef_df.head(10))


正向影響最大的前 10 個變數：
                           feature  coefficient
22  Type of Travel_Business travel     1.033953
7                  Online boarding     0.819265
20    Customer Type_Loyal Customer     0.704666
2            Inflight wifi service     0.529694
13                 Checkin service     0.411394
10                On-board service     0.379840
11                Leg room service     0.336359
24                  Class_Business     0.326682
15                     Cleanliness     0.292238
16      Departure Delay in Minutes     0.178260


In [79]:
print("\n負向影響最大的前 10 個變數：")
print(coef_df.tail(10))


負向影響最大的前 10 個變數：
                              feature  coefficient
0                                 Age    -0.124370
3   Departure/Arrival time convenient    -0.182585
4              Ease of Online booking    -0.188557
19                        Gender_Male    -0.302349
18                      Gender_Female    -0.334639
17           Arrival Delay in Minutes    -0.354176
25                          Class_Eco    -0.417576
26                     Class_Eco Plus    -0.546094
21    Customer Type_disloyal Customer    -1.341654
23     Type of Travel_Personal Travel    -1.670941


In [80]:
coef_abs_df = coef_df.copy()
coef_abs_df["abs_coefficient"] = coef_abs_df["coefficient"].abs()
coef_abs_df = coef_abs_df.sort_values(by="abs_coefficient", ascending=False)

print("影響力最大的前 15 個變數：")
print(coef_abs_df.head(15))

影響力最大的前 15 個變數：
                            feature  coefficient  abs_coefficient
23   Type of Travel_Personal Travel    -1.670941         1.670941
21  Customer Type_disloyal Customer    -1.341654         1.341654
22   Type of Travel_Business travel     1.033953         1.033953
7                   Online boarding     0.819265         0.819265
20     Customer Type_Loyal Customer     0.704666         0.704666
26                   Class_Eco Plus    -0.546094         0.546094
2             Inflight wifi service     0.529694         0.529694
25                        Class_Eco    -0.417576         0.417576
13                  Checkin service     0.411394         0.411394
10                 On-board service     0.379840         0.379840
17         Arrival Delay in Minutes    -0.354176         0.354176
11                 Leg room service     0.336359         0.336359
18                    Gender_Female    -0.334639         0.334639
24                   Class_Business     0.326682         0.3

# 結論：

### 本研究透過 Logistic Regression 分析顧客滿意度的影響因素，結果顯示：

### 旅遊型顧客（Personal Travel）與不忠誠顧客（disloyal customers）顯著較不滿意，而商務旅客則具有較高滿意度。

### 在服務層面：

### 線上登機（Online boarding）、機上 Wi-Fi 與整體服務品質對滿意度具有正向影響。

### 此外，航班延誤（Arrival Delay）對滿意度有明顯負面影響。

# test.csv 測試及資料

## 測試集資料讀入

In [82]:
df_test = pd.read_csv("/Users/apple/Desktop/Airline Passenger Satisfaction/test.csv")

## 資料預處理

In [83]:
df_test = df_test.drop(columns=["Unnamed: 0", "id"], errors="ignore")

df_test["satisfaction"] = df_test["satisfaction"].map({
    "satisfied": 1,
    "neutral or dissatisfied": 0
})

df_test["Arrival Delay in Minutes"] = df_test["Arrival Delay in Minutes"].fillna(
    df_test["Arrival Delay in Minutes"].median()
)

## 切分資料集

In [87]:
X_test_external = df_test.drop("satisfaction", axis=1)
y_test_external = df_test["satisfaction"]

## 直接套用剛剛訓練的模型 ><

In [88]:
y_pred_external = logistic_model.predict(X_test_external)

## 績效評分 Type 1/Type 2 Error

In [89]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("External Test Accuracy:", accuracy_score(y_test_external, y_pred_external))
print("\nClassification Report:")
print(classification_report(y_test_external, y_pred_external))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_external, y_pred_external))

External Test Accuracy: 0.8720357252848784

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.89     14573
           1       0.87      0.83      0.85     11403

    accuracy                           0.87     25976
   macro avg       0.87      0.87      0.87     25976
weighted avg       0.87      0.87      0.87     25976


Confusion Matrix:
[[13135  1438]
 [ 1886  9517]]
